In [12]:
import urllib.parse
import feedparser

def fetch_arxiv(query, max_results=100):
    base_url = "http://export.arxiv.org/api/query?"

    encoded_query = urllib.parse.quote(query)

    url = f"{base_url}search_query={encoded_query}&start=0&max_results={max_results}&sortBy=submittedDate&sortOrder=descending"

    print("DEBUG URL:", url)

    feed = feedparser.parse(url)

    papers = []
    for entry in feed.entries:

        # ✅ safer PDF extraction
        pdf_url = ""
        for link in entry.links:
            if link.type == "application/pdf":
                pdf_url = link.href

        papers.append({
            "id": entry.id,
            "title": entry.title.strip() if hasattr(entry, "title") else "",
            "authors": [author.name for author in entry.authors] if hasattr(entry, "authors") else [],
            "published": entry.published,
            "summary": entry.summary.strip() if hasattr(entry, "summary") else "",
            "pdf_url": pdf_url
        })

    return papers

In [13]:
from datetime import datetime

def filter_last_10_years(papers):
    current_year = datetime.now().year
    filtered = []

    for p in papers:
        try:
            year = int(p.get("published", "")[:4])
            if year >= current_year - 10:
                filtered.append(p)
        except:
            continue  # skip bad records

    return filtered

In [14]:
import requests
import time

def fetch_semantic_scholar(query, limit=100, api_key=None):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"

    params = {
        "query": query,
        "limit": limit,
        "fields": "title,authors,year,abstract,citationCount,url"
    }

    headers = {
        "User-Agent": "research-assistant"
    }

    if api_key:
        headers["x-api-key"] = api_key

    for attempt in range(3):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)

            if response.status_code == 200:
                data = response.json()

                papers = []
                for p in data.get("data", []):
                    papers.append({
                        "title": p.get("title"),
                        "authors": [a["name"] for a in p.get("authors", [])],
                        "year": p.get("year"),
                        "abstract": p.get("abstract"),
                        "citation_count": p.get("citationCount"),
                        "url": p.get("url")
                    })

                return papers

            elif response.status_code == 429:
                print("Rate limited... retrying")
                time.sleep(3)

            else:
                print(f"Error: {response.status_code}")
                break

        except requests.exceptions.RequestException as e:
            print("Request failed:", e)
            time.sleep(2)

    return []

In [21]:
from datetime import datetime
def is_ieee_paper(paper):
    text = (
        (paper.get("title") or "") +
        " " +
        (paper.get("abstract") or "")
    ).lower()

    keywords = [
        "ieee",
        "ieee conference",
        "ieee transactions",
        "ieee journal"
    ]

    return any(k in text for k in keywords)
if __name__ == "__main__":

    query = '(cat:cs.AI OR cat:cs.LG OR cat:cs.CL) AND (transformer OR "large language model" OR diffusion OR generative)'

    print("Fetching from arXiv...")
    arxiv_papers = fetch_arxiv(query, max_results=2000)

    print(f"Fetched: {len(arxiv_papers)} papers")

    # ✅ Clean + transform first
    cleaned_papers = []
    for p in arxiv_papers:
        try:
            cleaned_papers.append({
                "id": p.get("id"),
                "title": p.get("title", "").strip(),
                "authors": p.get("authors", []),
                "abstract": p.get("summary", "").strip(),
                "year": int(p.get("published", "")[:4]),
                "pdf_url": p.get("pdf_url", "")
            })
        except:
            continue  # skip bad records

    # ✅ Filter AFTER cleaning
    current_year = datetime.now().year
    cleaned_papers = [
    p for p in cleaned_papers
    if p["year"] >= current_year - 10
]

# 🔥 NEW: IEEE filter
    ieee_papers = [p for p in cleaned_papers if is_ieee_paper(p)]

    print(f"IEEE papers found: {len(ieee_papers)}")

    #print(f"After filtering (last 10 years): {len(cleaned_papers)}")

    # ✅ Safe print
    if cleaned_papers:
        print("\nSample Paper:")
        print(cleaned_papers[0])
    else:
        print("No papers found")

Fetching from arXiv...
DEBUG URL: http://export.arxiv.org/api/query?search_query=%28cat%3Acs.AI%20OR%20cat%3Acs.LG%20OR%20cat%3Acs.CL%29%20AND%20%28transformer%20OR%20%22large%20language%20model%22%20OR%20diffusion%20OR%20generative%29&start=0&max_results=2000&sortBy=submittedDate&sortOrder=descending
Fetched: 2000 papers
IEEE papers found: 2

Sample Paper:
{'id': 'http://arxiv.org/abs/2604.24758v1', 'title': 'Personalized Worked Example Generation from Student Code Submissions using Pattern-based Knowledge Components', 'authors': ['Griffin Pitts', 'Muntasir Hoq', 'Peter Brusilovsky', 'Narges Norouzi', 'Arto Hellas', 'Juho Leinonen', 'Bita Akram'], 'abstract': "Adaptive programming practice often relies on fixed libraries of worked examples and practice problems, which require substantial authoring effort and may not correspond well to the logical errors and partial solutions students produce while writing code. As a result, students may receive learning content that does not directly 

In [25]:
import urllib.parse
import feedparser
import json
import time
from datetime import datetime


# ==============================
# 1. FETCH FROM ARXIV (PAGINATION)
# ==============================
def fetch_arxiv(query, start=0, max_results=100):
    base_url = "http://export.arxiv.org/api/query?"

    encoded_query = urllib.parse.quote(query)

    url = (
        f"{base_url}"
        f"search_query={encoded_query}"
        f"&start={start}"
        f"&max_results={max_results}"
        f"&sortBy=submittedDate"
        f"&sortOrder=descending"
    )

    feed = feedparser.parse(url)

    papers = []

    for entry in feed.entries:
        try:
            pdf_url = ""
            for link in entry.links:
                if link.type == "application/pdf":
                    pdf_url = link.href

            papers.append({
                "id": entry.id,
                "title": entry.title.strip(),
                "authors": [a.name for a in entry.authors],
                "summary": entry.summary.strip(),
                "published": entry.published,
                "pdf_url": pdf_url
            })

        except:
            continue

    return papers


# ==============================
# 2. IEEE DETECTION (HEURISTIC)
# ==============================
def is_ieee_paper(paper):
    text = (
        (paper.get("title") or "") +
        " " +
        (paper.get("summary") or "")
    ).lower()

    keywords = [
        "ieee",
        "ieee conference",
        "ieee transactions",
        "ieee journal"
    ]

    return any(k in text for k in keywords)


# ==============================
# 3. MAIN PIPELINE
# ==============================
if __name__ == "__main__":

    queries = [
        'transformer',
        '"large language model"',
        'diffusion model',
        'generative AI',
        'deep learning',
        'machine learning'
    ]

    all_papers = []

    print("🚀 Fetching papers from arXiv...\n")

    # 🔥 MULTI-QUERY + PAGINATION
    for q in queries:
        print(f"🔍 Query: {q}")

        for start in range(0, 1000, 100):  # 10 pages per query
            batch = fetch_arxiv(q, start=start, max_results=100)

            if not batch:
                break

            all_papers.extend(batch)

            time.sleep(1)  # avoid overload

    print(f"\n📊 Total fetched (raw): {len(all_papers)}")


    # ==============================
    # 4. CLEAN + TRANSFORM
    # ==============================
    cleaned = []

    for p in all_papers:
        try:
            cleaned.append({
                "id": p["id"],
                "title": p["title"],
                "authors": p["authors"],
                "abstract": p["summary"],
                "year": int(p["published"][:4]),
                "pdf_url": p["pdf_url"]
            })
        except:
            continue


    # ==============================
    # 5. DEDUPLICATE
    # ==============================
    unique = {p["id"]: p for p in cleaned}
    cleaned = list(unique.values())

    print(f"📊 After deduplication: {len(cleaned)}")


    # ==============================
    # 6. FILTER LAST 10 YEARS
    # ==============================
    current_year = datetime.now().year

    recent_papers = [
        p for p in cleaned
        if p["year"] >= current_year - 10
    ]

    print(f"📊 Last 10 years: {len(recent_papers)}")


    # ==============================
    # 7. IEEE FILTER
    # ==============================
    ieee_papers = [
        p for p in recent_papers
        if is_ieee_paper(p)
    ]

    print(f"📊 IEEE-like papers: {len(ieee_papers)}")


    # ==============================
    # 8. SAVE DATASET
    # ==============================
    with open("ieee_papers_dataset.json", "w") as f:
        json.dump(ieee_papers, f, indent=2)

    print("\n✅ Dataset saved as ieee_papers_dataset.json")


    # ==============================
    # 9. SAMPLE OUTPUT
    # ==============================
    if ieee_papers:
        print("\n🔥 Sample Paper:\n")
        print(ieee_papers[0])
    else:
        print("No IEEE papers found")

🚀 Fetching papers from arXiv...

🔍 Query: transformer
🔍 Query: "large language model"
🔍 Query: diffusion model
🔍 Query: generative AI
🔍 Query: deep learning
🔍 Query: machine learning

📊 Total fetched (raw): 2000
📊 After deduplication: 1916
📊 Last 10 years: 1916
📊 IEEE-like papers: 0

✅ Dataset saved as ieee_papers_dataset.json
No IEEE papers found


In [26]:
import requests
import time
import json
import os

# 🔑 API KEY
API_KEY = "s2k-d2aox2poJehh8Q8eJBbgOeMuiBq40yG9E1UjrxWx"

HEADERS = {
    "x-api-key": API_KEY
}

BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/search"

FIELDS = "title,authors,year,venue,abstract,url"

QUERIES = [
    "artificial intelligence ieee",
    "machine learning ieee",
    "deep learning ieee",
    "transformer model ieee",
    "computer vision ieee",
    "nlp ieee"
]

REQUEST_DELAY = 1   # MUST (1 request/sec)
BATCH_SIZE = 100   # save every 1000 papers
TARGET = 400      # change 20000 / 40000

SAVE_FILE = "ieee_large_dataset.json"


# ✅ IEEE FILTER
def is_ieee(paper):
    venue = (paper.get("venue") or "").lower()
    return "ieee" in venue


# 💾 SAVE PROGRESS
def save_data(data):
    with open(SAVE_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


# 📥 LOAD IF EXISTS (resume)
def load_existing():
    if os.path.exists(SAVE_FILE):
        with open(SAVE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


# 🚀 FETCH FUNCTION
def fetch_large_dataset():
    all_papers = load_existing()
    seen_titles = set([p["title"] for p in all_papers])

    print(f"🔄 Resuming... Already have {len(all_papers)} papers")

    for query in QUERIES:
        offset = 0

        while len(all_papers) < TARGET:
            params = {
                "query": query,
                "limit": 100,   # max allowed
                "offset": offset,
                "fields": FIELDS
            }

            try:
                res = requests.get(BASE_URL, headers=HEADERS, params=params)

                if res.status_code != 200:
                    print("❌ Error:", res.text)
                    time.sleep(5)
                    continue

                data = res.json()
                papers = data.get("data", [])

                if not papers:
                    break

                for p in papers:
                    title = p.get("title")

                    if not title or title in seen_titles:
                        continue

                    if is_ieee(p):
                        all_papers.append(p)
                        seen_titles.add(title)

                offset += 100

                # 💾 SAVE PERIODICALLY
                if len(all_papers) % BATCH_SIZE == 0:
                    print(f"💾 Saved {len(all_papers)} papers")
                    save_data(all_papers)

                print(f"📊 Total collected: {len(all_papers)}")

                time.sleep(REQUEST_DELAY)

                if len(all_papers) >= TARGET:
                    break

            except Exception as e:
                print("⚠️ Error:", str(e))
                time.sleep(5)

    # FINAL SAVE
    save_data(all_papers)
    print(f"\n✅ DONE: {len(all_papers)} papers saved")


if __name__ == "__main__":
    fetch_large_dataset()

🔄 Resuming... Already have 0 papers
📊 Total collected: 20
📊 Total collected: 56
💾 Saved 100 papers
📊 Total collected: 100
📊 Total collected: 140
📊 Total collected: 202
📊 Total collected: 255
📊 Total collected: 293
📊 Total collected: 327
📊 Total collected: 365
📊 Total collected: 398
❌ Error: {"error":"Relevance search offset + limit must be < 1000. Consider '/paper/search/bulk' or the Datasets API to retrieve more papers."}

❌ Error: {"error":"Relevance search offset + limit must be < 1000. Consider '/paper/search/bulk' or the Datasets API to retrieve more papers."}

❌ Error: {"error":"Relevance search offset + limit must be < 1000. Consider '/paper/search/bulk' or the Datasets API to retrieve more papers."}

❌ Error: {"error":"Relevance search offset + limit must be < 1000. Consider '/paper/search/bulk' or the Datasets API to retrieve more papers."}

❌ Error: {"error":"Relevance search offset + limit must be < 1000. Consider '/paper/search/bulk' or the Datasets API to retrieve more pap

KeyboardInterrupt: 

In [1]:
import requests
import time
import json

API_KEY = "s2k-d2aox2poJehh8Q8eJBbgOeMuiBq40yG9E1UjrxWx"
import requests
import time
import json
import os


HEADERS = {"x-api-key": API_KEY}

URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"

FIELDS = "title,authors,year,venue,url,abstract"

# ✅ MULTIPLE QUERIES (IMPORTANT FOR LARGE DATA)
QUERIES = [
    "deep learning",
    "machine learning",
    "transformer models",
    "artificial intelligence",
    "natural language processing",
    "computer vision"
]

REQUEST_DELAY = 1  # API limit

# 💾 FILES
IEEE_FILE = "ieee_papers.json"
OTHER_FILE = "non_ieee_papers.json"
MIX_FILE = "all_papers.json"


# ✅ CHECK IEEE
def is_ieee(paper):
    return "ieee" in (paper.get("venue") or "").lower()


# 💾 SAVE FUNCTION
def save_json(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


# 🚀 MAIN FUNCTION
def fetch_all():
    ieee_papers = []
    other_papers = []
    all_papers = []

    seen_titles = set()

    for query in QUERIES:
        print(f"\n🔍 Query: {query}")

        token = None

        while True:
            params = {
                "query": query,
                "limit": 1000,
                "fields": FIELDS
            }

            if token:
                params["token"] = token

            response = requests.get(URL, headers=HEADERS, params=params)

            if response.status_code != 200:
                print("❌ Error:", response.text)
                time.sleep(5)
                continue

            data = response.json()

            papers = data.get("data", [])
            token = data.get("token")

            if not papers:
                break

            for p in papers:
                title = p.get("title")

                if not title or title in seen_titles:
                    continue

                paper_data = {
                    "title": title,
                    "authors": [a["name"] for a in p.get("authors", [])],
                    "year": p.get("year"),
                    "venue": p.get("venue"),
                    "url": p.get("url"),
                    "abstract": p.get("abstract") if p.get("abstract") else "Not available"
                }

                all_papers.append(paper_data)
                seen_titles.add(title)

                if is_ieee(p):
                    ieee_papers.append(paper_data)
                else:
                    other_papers.append(paper_data)

            print(f"📊 Total: {len(all_papers)} | IEEE: {len(ieee_papers)}")

            # 💾 SAVE EVERY 2000
            if len(all_papers) % 2000 == 0:
                print("💾 Saving progress...")
                save_json(all_papers, MIX_FILE)
                save_json(ieee_papers, IEEE_FILE)
                save_json(other_papers, OTHER_FILE)

            time.sleep(REQUEST_DELAY)

            if not token:
                break

    # FINAL SAVE
    save_json(all_papers, MIX_FILE)
    save_json(ieee_papers, IEEE_FILE)
    save_json(other_papers, OTHER_FILE)

    print("\n✅ DONE!")
    print(f"📊 Total Papers: {len(all_papers)}")
    print(f"📊 IEEE Papers: {len(ieee_papers)}")
    print(f"📊 Non-IEEE Papers: {len(other_papers)}")


# ▶️ RUN
if __name__ == "__main__":
    fetch_all()


🔍 Query: deep learning
📊 Total: 1000 | IEEE: 125
📊 Total: 1999 | IEEE: 256
📊 Total: 2998 | IEEE: 398
📊 Total: 3997 | IEEE: 540
📊 Total: 4996 | IEEE: 662
📊 Total: 5995 | IEEE: 798
📊 Total: 6995 | IEEE: 953
📊 Total: 7994 | IEEE: 1104
📊 Total: 8993 | IEEE: 1258
📊 Total: 9992 | IEEE: 1372
📊 Total: 10992 | IEEE: 1491
📊 Total: 11992 | IEEE: 1637
📊 Total: 12991 | IEEE: 1768
📊 Total: 13990 | IEEE: 1892
📊 Total: 14989 | IEEE: 2022
📊 Total: 15989 | IEEE: 2154
📊 Total: 16987 | IEEE: 2264
📊 Total: 17986 | IEEE: 2390
📊 Total: 18984 | IEEE: 2532
📊 Total: 19978 | IEEE: 2688
📊 Total: 20977 | IEEE: 2817
📊 Total: 21975 | IEEE: 2960
📊 Total: 22971 | IEEE: 3112
📊 Total: 23970 | IEEE: 3253
📊 Total: 24970 | IEEE: 3389
📊 Total: 25969 | IEEE: 3543
📊 Total: 26969 | IEEE: 3672
📊 Total: 27967 | IEEE: 3780
📊 Total: 28965 | IEEE: 3905
📊 Total: 29963 | IEEE: 4028
📊 Total: 30960 | IEEE: 4157
📊 Total: 31959 | IEEE: 4290
📊 Total: 32956 | IEEE: 4437
📊 Total: 33952 | IEEE: 4560
📊 Total: 34950 | IEEE: 4709
📊 Total: 3594

In [19]:
import pandas as pd

# Convert to DataFrame
df = pd.DataFrame(cleaned_papers)

# Save to CSV
df.to_csv("research_papers.csv", index=False, encoding="utf-8")

print("✅ Data saved to research_papers.csv")

✅ Data saved to research_papers.csv


In [ ]:
data\processed\raw\all_papers.json

In [ ]:
import ijson
import psycopg2
from psycopg2.extras import execute_batch
import time

conn = psycopg2.connect(
    host="localhost",
    database="rag_db",
    user="postgres",
    password="Harshu304@"
)

cursor = conn.cursor()

BATCH_SIZE = 1000
total_updates = 0
batch = []

start_time = time.time()

query = """
UPDATE papers
SET pdf_url = %s
WHERE LOWER(TRIM(title)) = LOWER(TRIM(%s));
"""

with open(r"C:\Users\harsh\agentic-research-assistant\data\raw\papers_raw.json", "r", encoding="utf-8") as f:

    papers = ijson.items(f, "item")

    for idx, paper in enumerate(papers, start=1):

        title = paper.get("title")
        pdf_url = paper.get("url")

        if title and pdf_url:
            batch.append((pdf_url, title))

        # progress log every 1000 records
        if idx % 1000 == 0:
            print(f"📄 Read {idx} papers")

        if len(batch) >= BATCH_SIZE:

            execute_batch(cursor, query, batch)

            conn.commit()

            total_updates += cursor.rowcount

            elapsed = time.time() - start_time

            print(
                f"✅ Batch Done | "
                f"Processed: {idx} | "
                f"Updated: {total_updates} | "
                f"Time: {elapsed:.2f}s"
            )

            batch = []

# remaining batch
if batch:

    execute_batch(cursor, query, batch)

    conn.commit()

    total_updates += cursor.rowcount

print("\n🎉 DONE")
print(f"✅ Total rows updated: {total_updates}")

cursor.close()
conn.close()

📄 Read 1000 papers


In [1]:
import ijson
import psycopg2
from psycopg2.extras import execute_values

conn = psycopg2.connect(
    host="localhost",
    database="rag_db",
    user="postgres",
    password="Harshu304@"
)

cursor = conn.cursor()

BATCH_SIZE = 10000
batch = []

# clear old temp data
cursor.execute("TRUNCATE temp_paper_urls;")
conn.commit()

with open(r"C:\Users\harsh\agentic-research-assistant\data\raw\papers_raw.json", "r", encoding="utf-8") as f:

    papers = ijson.items(f, "item")

    for idx, paper in enumerate(papers, start=1):

        title = paper.get("title")
        pdf_url = paper.get("url")

        if title and pdf_url:
            batch.append((title.strip().lower(), pdf_url))

        if len(batch) >= BATCH_SIZE:

            execute_values(
                cursor,
                """
                INSERT INTO temp_paper_urls (title, pdf_url)
                VALUES %s
                """,
                batch
            )

            conn.commit()

            print(f"✅ Inserted {idx} temp rows")

            batch = []

# remaining
if batch:

    execute_values(
        cursor,
        """
        INSERT INTO temp_paper_urls (title, pdf_url)
        VALUES %s
        """,
        batch
    )

    conn.commit()

print("🎉 Temp table load complete")

cursor.close()
conn.close()

✅ Inserted 10000 temp rows
✅ Inserted 20000 temp rows
✅ Inserted 30000 temp rows
✅ Inserted 40000 temp rows
✅ Inserted 50000 temp rows
✅ Inserted 60000 temp rows
✅ Inserted 70000 temp rows
✅ Inserted 80000 temp rows
✅ Inserted 90000 temp rows
✅ Inserted 100000 temp rows
✅ Inserted 110000 temp rows
✅ Inserted 120000 temp rows
✅ Inserted 130000 temp rows
✅ Inserted 140000 temp rows
✅ Inserted 150000 temp rows
✅ Inserted 160000 temp rows
✅ Inserted 170000 temp rows
✅ Inserted 180000 temp rows
✅ Inserted 190000 temp rows
✅ Inserted 200000 temp rows
✅ Inserted 210000 temp rows
✅ Inserted 220000 temp rows
✅ Inserted 230000 temp rows
✅ Inserted 240000 temp rows
✅ Inserted 250000 temp rows
✅ Inserted 260000 temp rows
✅ Inserted 270000 temp rows
✅ Inserted 280000 temp rows
✅ Inserted 290000 temp rows
✅ Inserted 300000 temp rows
✅ Inserted 310000 temp rows
✅ Inserted 320000 temp rows
✅ Inserted 330000 temp rows
✅ Inserted 340000 temp rows
✅ Inserted 350000 temp rows
✅ Inserted 360000 temp rows
✅

In [2]:
import ijson
import psycopg2
from psycopg2.extras import execute_values

conn = psycopg2.connect(
    host="localhost",
    database="rag_db",
    user="postgres",
    password="Harshu304@"
)

cursor = conn.cursor()

BATCH_SIZE = 10000
batch = []

# clear old temp data
cursor.execute("TRUNCATE temp_paper_authors;")
conn.commit()

with open(
    r"C:\Users\harsh\agentic-research-assistant\data\raw\papers_raw.json",
    "r",
    encoding="utf-8"
) as f:

    papers = ijson.items(f, "item")

    for idx, paper in enumerate(papers, start=1):

        title = paper.get("title")
        authors = paper.get("authors", [])

        # convert list -> string
        if isinstance(authors, list):
            authors = ", ".join(authors)

        if title and authors:
            batch.append(
                (
                    title.strip().lower(),
                    authors
                )
            )

        if len(batch) >= BATCH_SIZE:

            execute_values(
                cursor,
                """
                INSERT INTO temp_paper_authors (title, authors)
                VALUES %s
                """,
                batch
            )

            conn.commit()

            print(f"✅ Inserted {idx} temp rows")

            batch = []

# remaining batch
if batch:

    execute_values(
        cursor,
        """
        INSERT INTO temp_paper_authors (title, authors)
        VALUES %s
        """,
        batch
    )

    conn.commit()

print("🎉 Temp author table load complete")

cursor.close()
conn.close()

✅ Inserted 10093 temp rows
✅ Inserted 20199 temp rows
✅ Inserted 30308 temp rows
✅ Inserted 40396 temp rows
✅ Inserted 50502 temp rows
✅ Inserted 60595 temp rows
✅ Inserted 70697 temp rows
✅ Inserted 80784 temp rows
✅ Inserted 90878 temp rows
✅ Inserted 100971 temp rows
✅ Inserted 111062 temp rows
✅ Inserted 121166 temp rows
✅ Inserted 131251 temp rows
✅ Inserted 141349 temp rows
✅ Inserted 151435 temp rows
✅ Inserted 161521 temp rows
✅ Inserted 171617 temp rows
✅ Inserted 181706 temp rows
✅ Inserted 191805 temp rows
✅ Inserted 201894 temp rows
✅ Inserted 211977 temp rows
✅ Inserted 222058 temp rows
✅ Inserted 232154 temp rows
✅ Inserted 242238 temp rows
✅ Inserted 252324 temp rows
✅ Inserted 262399 temp rows
✅ Inserted 272494 temp rows
✅ Inserted 282590 temp rows
✅ Inserted 292679 temp rows
✅ Inserted 302761 temp rows
✅ Inserted 312859 temp rows
✅ Inserted 322962 temp rows
✅ Inserted 333041 temp rows
✅ Inserted 343118 temp rows
✅ Inserted 353196 temp rows
✅ Inserted 363292 temp rows
✅